In [2]:
from pyspark.sql import SparkSession
import os
import os
import sys

# Go to project root
project_root = os.path.abspath("/home/nhatquang/home-credit-credit-risk-model-stability")
os.chdir(project_root)
sys.path.insert(0, project_root)  # optional: to import modules from root

# --- Configurations ---
service_account_key = os.path.abspath("secrets/global-phalanx-449403-d2-aae80316b9df.json")
gcs_connector_path = os.path.abspath("data-platform/jars/gcs-connector-hadoop3-2.2.9-shaded.jar")

# Set environment variable (if not already set)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 6g --executor-memory 6g pyspark-shell"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = service_account_key
os.environ["PYSPARK_SUBMIT_ARGS"] = "--jars data-platform/jars/gcs-connector-hadoop3-2.2.9-shaded.jar pyspark-shell"
# Create Spark session with GCS configs
# Remove the 2 last config before push to production
spark = SparkSession.builder \
    .appName("ReadGCSCSV") \
    .master("local[*]") \
    .config("spark.jars", gcs_connector_path) \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", service_account_key) \
    .config("spark.sql.shuffle.partitions", "1") \
    .config("spark.default.parallelism", "1") \
    .getOrCreate()

25/07/28 18:13:12 WARN Utils: Your hostname, laboratory resolves to a loopback address: 127.0.1.1; using 192.168.1.42 instead (on interface enx00e04c3600c1)
25/07/28 18:13:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/07/28 18:13:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [ ]:
from pyspark.sql import SparkSession

# Path to your CSV in GCS
gcs_data_path = "gs://credit-risk-modeling-bucket/raw_data/"
gcs_train_file = "application_train.csv"
gcs_bureau_file = "bureau.csv"
gcs_bureau_balance_file = "bureau_balance.csv"
gcs_credit_card_balance = "credit_card_balance.csv"
gcs_installments_payments = "installments_payments.csv"
gcs_previous_application = "previous_application.csv"
gcs_pos_cash_balance = "POS_CASH_balance.csv"
gcs_connector_jars = "gs://credit-risk-modeling-bucket/jars/gcs-connector-hadoop3-latest.jar "

# spark = SparkSession.builder \
#         .master("local[*]") \
#         .appName("CreditRiskDataLoading") \
#         .config("spark.jars", gcs_connector_jars) \
#         .getOrCreate()

def load_csv_to_df(file_path, df_name): 
    """
    Helper function to load a CSV file into a Spark DataFrame and print info.
    """
    try:
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(file_path) \
            .sample(fraction=0.1, seed=42)
        print(f"✅ DataFrame '{df_name}' loaded successfully from GCS using Spark!")
        return df
    except Exception as e:
        print(f"❌ Error reading '{df_name}' from GCS: {e}")
        raise 
# Load all DataFrames
print("--- Loading application_train.csv ---")
train_df = load_csv_to_df(gcs_data_path + gcs_train_file, "train_df")

print("\n--- Loading bureau.csv ---")
bureau_df = load_csv_to_df(gcs_data_path + gcs_bureau_file, "bureau_df")

print("\n--- Loading bureau_balance.csv ---")
bureau_balance_df = load_csv_to_df(gcs_data_path + gcs_bureau_balance_file, "bureau_balance_df")

print("\n--- Loading POS_CASH_balance.csv ---")
pos_cash_balance_df = load_csv_to_df(gcs_data_path + gcs_pos_cash_balance, "pos_cash_balance_df")

print("\n--- Loading credit_card_balance.csv ---")
credit_card_balance_df = load_csv_to_df(gcs_data_path + gcs_credit_card_balance, "credit_card_balance_df")

print("\n--- Loading installments_payments.csv ---")
installments_payments_df = load_csv_to_df(gcs_data_path + gcs_installments_payments, "installments_payments_df")

print("\n--- Loading previous_application.csv ---")
previous_application_df = load_csv_to_df(gcs_data_path + gcs_previous_application, "previous_application_df")


# spark.stop()

--- Loading application_train.csv ---


# Joining Dataframe

In [ ]:
from pyspark.sql.functions import col

def prefix_df(df, prefix):
    new_cols = [prefix + c for c in df.columns]
    return df.toDF(*new_cols)

t = prefix_df(train_df, "t_")
b = prefix_df(bureau_df, "b_")
bb = prefix_df(bureau_balance_df, "bb_")
pa = prefix_df(previous_application_df, "pa_")

# Now you can join safely using key t_SK_ID_CURR == b_SK_ID_CURR, etc.
application_df = t.join(b, col("t_SK_ID_CURR") == col("b_SK_ID_CURR"), "left") \
    .join(bb, col("b_SK_ID_BUREAU") == col("bb_SK_ID_BUREAU"), "left") \
    .join(pa, col("t_SK_ID_CURR") == col("pa_SK_ID_CURR"), "left")


# Try simple spark ml

In [ ]:
from pyspark.sql.functions import col, when, isnan, count
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
application_dtypes = application_df.dtypes
cat_cols = [name for name, dtype in application_dtypes if dtype == "string"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
# create one-hot encoders
encoders = [OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_vec"], dropLast=False) for c in cat_cols]
num_cols = [c for c in application_df.columns if c not in cat_cols + ['TARGET', 'SK_ID_CURR']]
assembler = VectorAssembler(
    inputCols=[f"{c}_vec" for c in cat_cols] + num_cols,
    outputCol="features"
)

In [ ]:

dt = DecisionTreeClassifier(labelCol="TARGET", featuresCol="features")

pipeline = Pipeline(stages=indexers + encoders + [assembler, dt])

# Cross-validation setup
paramGrid = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [5, 10]) \
    .addGrid(dt.minInstancesPerNode, [1, 5]) \
    .build()

cv = CrossValidator(
    estimator=pipeline,
    evaluator=BinaryClassificationEvaluator(labelCol="TARGET"),
    estimatorParamMaps=paramGrid,
    numFolds=3
)

cvModel = cv.fit(application_df)
result = cvModel.transform(application_df)


25/07/28 18:09:52 WARN GoogleCloudStorageReadChannel: Failed read retry #1/10 for 'gs://credit-risk-modeling-bucket/raw_data/application_train.csv'. Sleeping...
java.net.SocketTimeoutException: Read timed out
	at java.base/sun.nio.ch.NioSocketImpl.timedRead(NioSocketImpl.java:288)
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:314)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/sun.security.ssl.SSLSocketInputRecord.read(SSLSocketInputRecord.java:484)
	at java.base/sun.security.ssl.SSLSocketInputRecord.readHeader(SSLSocketInputRecord.java:478)
	at java.base/sun.security.ssl.SSLSocketInputRecord.bytesInCompletePacket(SSLSocketInputRecord.java:70)
	at java.base/sun.security.ssl.SSLSocketImpl.readApplicationRecord(SSLSocketImpl.java:1465)
	at java.base/sun.security.ssl.SSLSocketImpl$AppInputStream.

KeyboardInterrupt: 

In [ ]:
spark.stop()

25/07/28 18:12:18 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-3c5b022a-8278-4cd5-a89c-7927f31bb171/3b/temp_shuffle_ea6f0885-e38a-4440-ae15-1a475768f4ab
25/07/28 18:12:18 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-3c5b022a-8278-4cd5-a89c-7927f31bb171/38/temp_shuffle_ccba5940-3ecb-4936-bb02-e19774bf7274
25/07/28 18:12:18 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-3c5b022a-8278-4cd5-a89c-7927f31bb171/0a/temp_shuffle_be2d3b9d-7aa2-49de-8364-d0294272196f
25/07/28 18:12:18 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-3c5b022a-8278-4cd5-a89c-7927f31bb171/10/temp_shuffle_969542b7-8aa3-4da7-b7d1-bcbceba29b3d
25/07/28 18:12:18 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-3c5b022a-8278-4cd5-a89c-7927f31bb171/26/temp_shuffle_93ddc03b-550d-42b1-b305-466103a2cddb
25/07/28 18:12:18 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-3c5b022a-8278-4cd5-a89c-7927f31bb171/01/temp_shuffle_1c40b844-ca87-459c-9e78-0daea679d15b
25/07/28 1

25/07/28 18:12:27 WARN GoogleCloudStorageReadChannel: Failed read retry #1/10 for 'gs://credit-risk-modeling-bucket/raw_data/bureau.csv'. Sleeping...
java.net.SocketTimeoutException: Read timed out
	at java.base/sun.nio.ch.NioSocketImpl.timedRead(NioSocketImpl.java:288)
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:314)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/sun.security.ssl.SSLSocketInputRecord.read(SSLSocketInputRecord.java:484)
	at java.base/sun.security.ssl.SSLSocketInputRecord.readFully(SSLSocketInputRecord.java:467)
	at java.base/sun.security.ssl.SSLSocketInputRecord.decodeInputRecord(SSLSocketInputRecord.java:243)
	at java.base/sun.security.ssl.SSLSocketInputRecord.decode(SSLSocketInputRecord.java:181)
	at java.base/sun.security.ssl.SSLTransport.decode(SSLTransport.java:111)
	at